In [1]:
# Required Python packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
from scipy.stats import bernoulli
from scipy.optimize import fmin_slsqp as min

import tensorflow
import tensorflow.keras as keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.optimizers as opt
from tensorflow.keras import backend as keras_backend

# import keras
# from keras.models import Sequential
# from keras.layers import Dense
# from keras import layers
# from keras.callbacks import EarlyStopping
# import keras.optimizers as opt
# from keras import backend as keras_backend

import matplotlib.pyplot as plt
import math


#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import get_all_monthly_option_expiries, \
                                                     find_last_thurs_date_of_month, \
                                                     prev_workday_if_holiday, prev_weekday, find_wkly_expries,\
                                                     date_of_prev_thurs

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data

#import traded options parameters and info
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_strikes, generate_mthly_strikes

#Import pricing functions
from ipynb.fs.full.user_defined_vik_functions import BSM_call_vec, BSM_put_vec

#Import implied volatility processing functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_iv, gen_interpolated_iv

#Import Risk-free interest rate generation functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_ir

#Import Stochatic process simulation functions
from ipynb.fs.full.user_defined_vik_functions import gbm1W_simulation, generate_covariance_from_correlation


/opt/tljh/user/lib/python3.6/site-packages/tensorboard/compat/tensorflow_stub/dtypes.py:541: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/opt/tljh/user/lib/python3.6/site-packages/tensorboard/compat/tensorflow_stub/dtypes.py:542: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/opt/tljh/user/lib/python3.6/site-packages/tensorboard/compat/tensorflow_stub/dtypes.py:543: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/opt/tljh/user/lib/python3.6/site-packages/tensorboard/compat/tensorflow_stub/dtypes.

In [2]:
# Stock Index of Interest
stock_ident = "BANKNIFTY"

# The scope of this code is to hedge one option 
# and scope will be extended to a portfolio
no_of_assets = 1
cor_mat = [[1]]

#Path to refer data
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "ATM"

#Product type to hedge: either "CE" or "PE"
prod_type = "CE"

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2020: [3]}

#List of holidays
holidays_list = [date(2020, 3, 10)]

#Simulation Parameters
no_of_paths = 5000
#Initialisation of Neural Network hyperparameters for pre-training network
batch_size_divisor = 10
no_of_epochs = 1000 # keep 10 for debugging coding errors
no_of_output_nodes = no_of_assets
batch_size = int(no_of_paths/batch_size_divisor)


In [3]:
#Function to price European call options
def pricer_vanilla_by_nn(sim_stock_mat, option_type, opt_strike, dt, \
                         r, vol, nnet_model, batch_size, no_of_epochs,):

    # Current stock price is at index 0. This extracts stock price at the next time step
    stock_vec = sim_stock_mat[:, 1]
        
    # The following steps extract option value/pay-off at the next week
    if (option_type == "CE"):
        option_value = BSM_call_vec(stock_vec, opt_strike, r, vol, dt)
        option_value = np.asarray(option_value)
        option_value = option_value.reshape(-1, 1)
    else:
        option_value = BSM_put_vec(stock_vec, opt_strike, r, vol, dt)
        option_value = np.asarray(option_value)
        option_value = option_value.reshape(-1, 1)
        
    #Creating the training data for the neural network model
    # Neural Network Function using RELU activaion function
    # Build Neural Network model(using Stock prices at next exercise time step and their option value)   
    X_train = stock_vec
    X_train = X_train.reshape(-1, 1)

    Y_train = option_value
    Y_train = np.asarray(Y_train)
    Y_train.reshape(-1, 1)

    nnet_output = nnet_model.fit(X_train, Y_train, epochs=no_of_epochs, batch_size=batch_size, 
                                 verbose=1, validation_split=0.3)

    pfl_weights = np.array(nnet_model.layers[1].get_weights())
            
    return pfl_weights


def pre_train_nn(sim_stock_mat, option_type, \
                 opt_strike, dt, \
                 nnet_model, batch_size, no_of_epochs):

    stock_vec = sim_stock_mat[:, 1]
        
    # The following steps extract option value/pay-off at the next time step
    if (option_type == "CE"):
        option_value = np.maximum(stock_vec - opt_strike, 0)
    else:
        option_value = np.maximum(opt_strike - stock_vec, 0)
        
    option_value = np.asarray(option_value)
    option_value = option_value.reshape(-1, 1)
        
    #Creating the training data for the neural network model
    # Neural Network Function using RELU activaion function
    # Build Neural Network model(using Stock prices at next exercise time step and their option value)
    X_train = stock_vec
    X_train = X_train.reshape(-1, 1)

    Y_train = option_value
    Y_train = np.asarray(Y_train)
    Y_train.reshape(-1, 1, 1)

    nnet_output = nnet_model.fit(X_train, Y_train, epochs=no_of_epochs, batch_size=batch_size, verbose=0, validation_split=0.3)

    return nnet_model


In [4]:
#########################
#User-defined functions #
#########################

def generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, \
                                       dict_wkly_expiries_each_month, \
                                       dict_ce_wkly_strikes, dict_pe_wkly_strikes, \
                                       dict_wkly_spots, dict_wkly_iv, \
                                       prod_type, prod_moneyness, output_path, \
                                       option_type = "CE", no_of_assets=1, cor_mat=[[1]],\
                                       batch_size_divisor=10, no_of_epochs = 1000):
    ce_pfl_weights = []
    pe_pfl_weights = [] 
    cash_weights = []
    for each_month in mthly_expiries_list:
        K = dict_mthly_strikes[each_month.strftime("%d-%b-%Y")]
        weekly_pfl_weights = []
        ce_dict = {}
        pe_dict = {}
        cash_dict = {}
        for week in range(0, 4):
            s_t = dict_wkly_spots[each_month.strftime("%d-%b-%Y")][week]
            vol_list = np.array([dict_wkly_iv[each_month.strftime("%d-%b-%Y")][week]])
            r = np.array([dict_wkly_ir[each_month.strftime("%d-%b-%Y")][week]])
            
            if (week == 3):
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = each_month
            else:
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week + 1]
         
            sim_stock_mat = gbm1W_simulation(no_of_paths, sim_start_date, sim_end_date, \
                                             no_of_assets, s_t, r, vol_list, cor_mat)
            
            call_strikes_hdge_list_pos = np.sort(np.array(dict_ce_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).tolist()
            call_strikes_hdge_list = [-1 * strike for strike in call_strikes_hdge_list_pos]
#             put_strikes_hdge_list = np.sort(np.array(dict_pe_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).tolist()
#             no_of_hidden_nodes = len(call_strikes_hdge_list) + len(put_strikes_hdge_list)
            no_of_hidden_nodes = len(call_strikes_hdge_list)

            call_weights = list(np.ones(len(call_strikes_hdge_list)))
#             put_weights = list(-1 * np.ones(len(put_strikes_hdge_list)))
#             static_layer_weights = call_weights + put_weights
            static_layer_weights = call_weights 
#             static_node_weights = call_strikes_hdge_list + put_strikes_hdge_list
            static_node_weights = call_strikes_hdge_list
            static_weights = [np.array([static_layer_weights]), np.array(static_node_weights)]

            nnet_model = Sequential()
            hidden_layer = Dense(no_of_hidden_nodes, activation='relu', input_dim=no_of_assets)
            hidden_layer.trainable = False
            nnet_model.add(hidden_layer)
            nnet_model.layers[0].set_weights(static_weights) 
#             output_layer = Dense(no_of_output_nodes, activation='linear', kernel_initializer='normal', use_bias=True, kernel_constraint=keras.constraints.NonNeg())
            
            outer_initializer = tensorflow.keras.initializers.Constant(value=(1/no_of_hidden_nodes))
            output_layer = Dense(no_of_output_nodes, activation='linear', kernel_initializer=outer_initializer, use_bias=True)
            
            output_layer.trainable = True
            nnet_model.add(output_layer)
            nnet_model.compile(optimizer=opt.Adam(lr=0.001), loss='mean_squared_error')
###################
            # Adjusted dt to accommodate the time to maturity of option and not simulation time
###################
#             dt = float((sim_end_date - sim_start_date).days) / 365
            dt = float((each_month - sim_end_date).days) / 365
            batch_size = int(no_of_paths/batch_size_divisor)
            nnet_model = pre_train_nn(sim_stock_mat, option_type, \
                                      K, dt, nnet_model, batch_size, no_of_epochs=100)
        
            #Final Params adjustment for the final neural networks
            keras_backend.set_value(nnet_model.optimizer.lr, 5e-4)

            ######################################################################################
            ### Portfolio weights of shorter term options from neural network of European options ###
            ######################################################################################

            pfl_weights = pricer_vanilla_by_nn(sim_stock_mat, option_type, K, dt, \
                          r, vol_list[0], nnet_model, batch_size, no_of_epochs)
            
            ce_dates_list = [sim_start_date.strftime("%d-%b-%Y") for i in range(0, len(call_strikes_hdge_list))]
            ce_wt_list = np.array(pfl_weights[0]).reshape(-1)[0:len(call_strikes_hdge_list)]
            ce_dict = {"Date": ce_dates_list, "Strike": call_strikes_hdge_list_pos, "Pfl_weights": ce_wt_list}
            df_ce = pd.DataFrame(ce_dict)
            ce_pfl_weights.append(df_ce)
            
#             pe_wt_list = np.array(pfl_weights[0]).reshape(-1)[len(call_strikes_hdge_list):]
#             pe_dates_list = [sim_start_date.strftime("%d-%b-%Y") for i in range(0, len(put_strikes_hdge_list))]
#             pe_dict = {"Date": pe_dates_list, "Strike":put_strikes_hdge_list, "Pfl_weights": pe_wt_list}
#             df_pe = pd.DataFrame(pe_dict)
#             pe_pfl_weights.append(df_pe)
            
            cash_date = [sim_start_date.strftime("%d-%b-%Y")]
            cash_weight = np.array(pfl_weights[1][0])
            cash_dict = {"Date":cash_date, "Pfl_weights":cash_weight} 
            df_cash = pd.DataFrame(cash_dict)
            cash_weights.append(df_cash)
            
        df_ce_pfl_weights = pd.concat(ce_pfl_weights, axis=0)
#         df_pe_pfl_weights = pd.concat(pe_pfl_weights, axis=0)
        df_cash_pfl_weights = pd.concat(cash_weights, axis=0)
    
        df_ce_pfl_weights.to_csv(output_path  + "B2_" + prod_moneyness + "_" + "CE_Pfl_weights" + ".csv", index = False)
#         df_pe_pfl_weights.to_csv(output_path + "B1_" + prod_moneyness + "_" + "PE_Pfl_weights" + ".csv", index = False)
        df_cash_pfl_weights.to_csv(output_path  + "B2_" +  prod_moneyness + "_" + "Cash_Pfl_weights" + ".csv", index = False)
    
    return df_ce_pfl_weights, df_cash_pfl_weights

def output_strikes_hedged(prod_type, dict_mthly_strikes, output_path):
    list_of_dates = []
    list_of_prod_type = []
    list_of_strikes = []
    for each_month in dict_mthly_strikes.keys():
        list_of_dates.append(each_month)
        list_of_strikes.append(dict_mthly_strikes[each_month])
        list_of_prod_type.append(prod_type)
    dict_hedged_prod = {"Date":list_of_dates, "prod_type":list_of_prod_type, "Strike":list_of_strikes}
    df_hedged_prod = pd.DataFrame(dict_hedged_prod) 
    df_hedged_prod.to_csv(output_path + "B2_" + prod_type + "_" + "strikes_hedged.csv", index = False)
    return(df_hedged_prod)


In [5]:

#Find the monthly strikes of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)

#Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, input_data_path, prod_type_lists=["FUT", "CE", "PE"], stock_ident=stock_ident)

#Load weekly strikes of short term options used for hedging monthly options 
dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots = generate_weekly_strikes(dict_wkly_expiries_each_month, input_data_path, stock_ident = "BANKNIFTY")     

#Load monthly strikes of interest as of required moneyness on the begining of month
dict_mthly_strikes = generate_mthly_strikes(mthly_mkt_data, prod_moneyness, prod_type, holidays_list, stock_ident=stock_ident)

#Load weekly IV for Black-Scholes model input and monte-carlo simulation of stock paths
dict_wkly_iv = generate_weekly_iv(dict_wkly_expiries_each_month, dict_wkly_spots, dict_mthly_strikes, prod_type, output_path=output_data_path)

#Load weekly risk free interest rates from futures
#This function to be changed based on updates on risk-free-interest - now kept it as cosntant
dict_wkly_ir = generate_weekly_ir(dict_wkly_expiries_each_month, mthly_mkt_data, stock_ident)

#Load Static portfolio weights built by neural network
ce_pfl_wts, cash_pfl_wts = generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, dict_wkly_expiries_each_month, \
                                                        dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots, \
                                                        dict_wkly_iv, prod_type, prod_moneyness, output_data_path, \
                                                        option_type = prod_type, no_of_assets=no_of_assets, cor_mat=cor_mat,\
                                                        batch_size_divisor=batch_size_divisor, no_of_epochs = no_of_epochs) 

# The strike correspnding to the option that is hedged
df_hedged_prod = output_strikes_hedged(prod_type, dict_mthly_strikes, output_data_path)



ipynb.fs.full.user_defined_vik_functions:301: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set

Epoch 62/1000
7/7 [==============================] - 0s 7ms/step - loss: 738.7124 - val_loss: 709.0574
Epoch 63/1000
7/7 [==============================] - 0s 7ms/step - loss: 716.0421 - val_loss: 686.2272
Epoch 64/1000
7/7 [==============================] - 0s 7ms/step - loss: 694.9979 - val_loss: 666.5396
Epoch 65/1000
7/7 [==============================] - 0s 7ms/step - loss: 673.8418 - val_loss: 646.3287
Epoch 66/1000
7/7 [==============================] - 0s 7ms/step - loss: 654.8978 - val_loss: 628.5812
Epoch 67/1000
7/7 [==============================] - 0s 8ms/step - loss: 634.5939 - val_loss: 609.5270
Epoch 68/1000
7/7 [==============================] - 0s 7ms/step - loss: 615.2051 - val_loss: 590.3598
Epoch 69/1000
7/7 [==============================] - 0s 7ms/step - loss: 596.9110 - val_loss: 573.8436
Epoch 70/1000
7/7 [==============================] - 0s 7ms/step - loss: 579.6016 - val_loss: 556.0753
Epoch 71/1000
7/7 [==============================] - 0s 8ms/step - loss: 

7/7 [==============================] - 0s 7ms/step - loss: 17.3803 - val_loss: 17.5400
Epoch 222/1000
7/7 [==============================] - 0s 8ms/step - loss: 17.1011 - val_loss: 17.2702
Epoch 223/1000
7/7 [==============================] - 0s 7ms/step - loss: 16.8336 - val_loss: 17.0315
Epoch 224/1000
7/7 [==============================] - 0s 7ms/step - loss: 16.5842 - val_loss: 16.7641
Epoch 225/1000
7/7 [==============================] - 0s 7ms/step - loss: 16.3418 - val_loss: 16.5005
Epoch 226/1000
7/7 [==============================] - 0s 7ms/step - loss: 16.0836 - val_loss: 16.2560
Epoch 227/1000
7/7 [==============================] - 0s 8ms/step - loss: 15.8510 - val_loss: 16.0408
Epoch 228/1000
7/7 [==============================] - 0s 20ms/step - loss: 15.6329 - val_loss: 15.8101
Epoch 229/1000
7/7 [==============================] - 0s 7ms/step - loss: 15.3704 - val_loss: 15.5785
Epoch 230/1000
7/7 [==============================] - 0s 7ms/step - loss: 15.1340 - val_loss: 15

7/7 [==============================] - 0s 7ms/step - loss: 2.4326 - val_loss: 2.4650
Epoch 385/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.4050 - val_loss: 2.4351
Epoch 386/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.3601 - val_loss: 2.4115
Epoch 387/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.3271 - val_loss: 2.3769
Epoch 388/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.2981 - val_loss: 2.3512
Epoch 389/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.2734 - val_loss: 2.3209
Epoch 390/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.2481 - val_loss: 2.2981
Epoch 391/1000
7/7 [==============================] - 0s 9ms/step - loss: 2.2192 - val_loss: 2.2676
Epoch 392/1000
7/7 [==============================] - 0s 19ms/step - loss: 2.1931 - val_loss: 2.2443
Epoch 393/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.1675 - val_loss: 2.2140
Epoch 394/1000

...

Epoch 495/1000
7/7 [==============================] - 0s 7ms/step - loss: 1.8942 - val_loss: 0.5563
Epoch 496/1000
7/7 [==============================] - 0s 8ms/step - loss: 2.8306 - val_loss: 0.4220
Epoch 497/1000
7/7 [==============================] - 0s 7ms/step - loss: 3.1039 - val_loss: 1.0029
Epoch 498/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.5046 - val_loss: 0.8829
Epoch 499/1000
7/7 [==============================] - 0s 20ms/step - loss: 2.2353 - val_loss: 0.5572
Epoch 500/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.8103 - val_loss: 1.4390
Epoch 501/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.5269 - val_loss: 1.0117
Epoch 502/1000
7/7 [==============================] - 0s 8ms/step - loss: 2.0248 - val_loss: 1.3312
Epoch 503/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.2422 - val_loss: 0.8507
Epoch 504/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.0903 - val_loss: 0.6626

Epoch 577/1000
7/7 [==============================] - 0s 7ms/step - loss: 3.8958 - val_loss: 4.8310
Epoch 578/1000
7/7 [==============================] - 0s 7ms/step - loss: 5.9600 - val_loss: 5.7363
Epoch 579/1000
7/7 [==============================] - 0s 7ms/step - loss: 6.9626 - val_loss: 4.2704
Epoch 580/1000
7/7 [==============================] - 0s 7ms/step - loss: 3.9072 - val_loss: 4.6635
Epoch 581/1000
7/7 [==============================] - 0s 7ms/step - loss: 3.2068 - val_loss: 1.9450
Epoch 582/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.5441 - val_loss: 1.2750
Epoch 583/1000
7/7 [==============================] - 0s 7ms/step - loss: 1.9967 - val_loss: 2.3911
Epoch 584/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.5424 - val_loss: 3.1615
Epoch 585/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.6617 - val_loss: 3.7167
Epoch 586/1000
7/7 [==============================] - 0s 7ms/step - loss: 3.2993 - val_loss: 10.2618

Epoch 659/1000
7/7 [==============================] - 0s 9ms/step - loss: 2.0774 - val_loss: 0.7994
Epoch 660/1000
7/7 [==============================] - 0s 8ms/step - loss: 3.4709 - val_loss: 6.1940
Epoch 661/1000
7/7 [==============================] - 0s 9ms/step - loss: 8.2353 - val_loss: 6.4436
Epoch 662/1000
7/7 [==============================] - 0s 8ms/step - loss: 8.1706 - val_loss: 1.5164
Epoch 663/1000
7/7 [==============================] - 0s 7ms/step - loss: 12.5910 - val_loss: 16.0112
Epoch 664/1000
7/7 [==============================] - 0s 8ms/step - loss: 9.9607 - val_loss: 0.6860
Epoch 665/1000
7/7 [==============================] - 0s 8ms/step - loss: 2.9449 - val_loss: 1.5071
Epoch 666/1000
7/7 [==============================] - 0s 8ms/step - loss: 6.2549 - val_loss: 3.4808
Epoch 667/1000
7/7 [==============================] - 0s 21ms/step - loss: 4.6739 - val_loss: 2.6006
Epoch 668/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.0031 - val_loss: 1.70

7/7 [==============================] - 0s 7ms/step - loss: 5.4106 - val_loss: 0.7716
Epoch 822/1000
7/7 [==============================] - 0s 7ms/step - loss: 3.7359 - val_loss: 1.6297
Epoch 823/1000
7/7 [==============================] - 0s 7ms/step - loss: 5.9620 - val_loss: 16.7592
Epoch 824/1000
7/7 [==============================] - 0s 7ms/step - loss: 12.6066 - val_loss: 4.9291
Epoch 825/1000
7/7 [==============================] - 0s 7ms/step - loss: 5.1588 - val_loss: 8.9967
Epoch 826/1000
7/7 [==============================] - 0s 7ms/step - loss: 4.9004 - val_loss: 11.6912
Epoch 827/1000
7/7 [==============================] - 0s 7ms/step - loss: 19.9679 - val_loss: 4.4648
Epoch 828/1000
7/7 [==============================] - 0s 7ms/step - loss: 4.8068 - val_loss: 3.9641
Epoch 829/1000
7/7 [==============================] - 0s 7ms/step - loss: 6.8674 - val_loss: 1.7690
Epoch 830/1000
7/7 [==============================] - 0s 7ms/step - loss: 7.6106 - val_loss: 2.5649
Epoch 831/1

Epoch 903/1000
7/7 [==============================] - 0s 7ms/step - loss: 2.3597 - val_loss: 1.4489
Epoch 904/1000
7/7 [==============================] - 0s 7ms/step - loss: 3.4215 - val_loss: 7.8708
Epoch 905/1000
7/7 [==============================] - 0s 7ms/step - loss: 7.0347 - val_loss: 42.4792
Epoch 906/1000
7/7 [==============================] - 0s 7ms/step - loss: 22.7915 - val_loss: 0.1574
Epoch 907/1000
7/7 [==============================] - 0s 7ms/step - loss: 3.5157 - val_loss: 4.7427
Epoch 908/1000
7/7 [==============================] - 0s 19ms/step - loss: 3.9179 - val_loss: 1.1366
Epoch 909/1000
7/7 [==============================] - 0s 7ms/step - loss: 3.2582 - val_loss: 2.7663
Epoch 910/1000
7/7 [==============================] - 0s 7ms/step - loss: 3.5014 - val_loss: 5.1440
Epoch 911/1000
7/7 [==============================] - 0s 7ms/step - loss: 5.9676 - val_loss: 0.8138
Epoch 912/1000
7/7 [==============================] - 0s 7ms/step - loss: 11.3523 - val_loss: 6.8